# HA#5 - Sarcasm Detection with DistilBERT

CSC620 Natural Langauge Technologies
Author: Ryan Alvarado

## What are we doing, and why?
HA#3 was built on Naïve Bayes classifier. It works by counting word frequencies and using Bayes' theorem to estimate the probability that a headline is sarcastic. It treats every word independentently with no concept of context.

In this project we are using DistilBERT, a transformer-based language model. Transformers ready every word in relation to every other word in the sentence. This model can pick up on subtle cues such as irony, tone, and constract.

Fine-tuning means we start from a model that already knows English and train it for a short time on our specific task. 

## Concepts used
**Transformer embeddings**
Each token is represented as a dense vector of numbers that encode meaning and context

**CLS token**
A special token prepended to every input. After passing through all transofmrer layers its embedding summarises the entire sentence.

**Classification head**
A simple linear layer (of matrix multiplication) that maps the CLS vector to one score per class

**Fine-tuning**
Continuing training on task-specific data so the pre-training weights specialize to our problem

### Data Flow
```
Raw headline text
        |
        v
Tokenizer  ->  token IDs + attention mask
        |
        v
DistilBERT (6 transformer layers)  ->  contextual embedding for every token
        |
        v
[CLS] token embedding  ->  single vector summarising the whole sentence
        |
        v
Classification head (Linear 768 -> 2)  ->  logits [score_0, score_1]
        |
        v
argmax  ->  predicted label (0 = not sarcastic, 1 = sarcastic)
```

In [ ]:
# HA #5 pt. 2: Fine-tuning DistilBERT for Sarcasm detection

## Step 1: Install dependencies

In [ ]:
!pip install transformers datasets scikit-learn torch --quiet

## Imports

In [ ]:
import json         # to read .json dataset
import pandas as pd # for tabular data manipulation & display
import numpy as nmp # for numerical operations (argmax on logits, etc.)

# train_test_split: randomly divides data into a training set an da held-out test set
from sklearn.model_selection import train_test_split

# Accuracy - overall fraction correct (can be misleading on bad data)
# Precision - of everything the model labeled sarcastic, how many truly were?
# Recall - of all the truly sarcastic headlines, how many did the model catch?
# F1 - harmonic mean of precision and recall, balances both concerns
from sklearn.metrics import (
    accuract_scores,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Dataset - abstracy PyTorch class; we subclass it to define how to fetch one example
import torch
from torch.utils.data import Dataset


# DistilVeryTokenizerFast - concerts raw txt into token IDs
# DistilVeryForSequnceClassification - DistilBERT backbone + classification head
# Trainer / TrainingArguments - Hugging Face high-level training loop
from transformers import (
    DilstilBertTokenizerFast,
    DistilBertForSequenceClassificationl
    Trainer,
    TrainingArguments,
)

### Step 3: Load the Dataset

**Before running this cell:**
1. Download `Sarcasm_Headlines_Dataset.json` from https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection
2. Upload it to the Colab session using the Files panel on the left sidebar.

In [ ]:
# The file is newline-delimited JSON. Each line is a complete JSON object.
# We can't use json.load() on the whole file at once, so we go line-by-line.

records = []
with open('Sarcasm_Headlines_Dataset.json', 'r') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

# Dataframe has three columns:
#   article_link - URL to the original article (ignored)
#   headline - news headline txt (our input x)
#   is_sarcastic:    1 = sarcastic, 0 = non-sarcastic (label y)

print(df.head())
print(f"\nTotal samples: {len(df)}")
print(f"Sarcastic: {df['is_sarcastic'].sum()}")
print(f"Not sarcastic: {(df['is_sarcastic'] == 0).sum()}")

## Train / Test Split

In [ ]:
# Pull text and labels out of the DataFrame as plain Python list.
# The tokenizer and Dataset class expects lists, not pandas Series.

text = df['headline'].tolist()
label = df['is_sarcastic'].tolist()

# Split 80% train / 20% test
# random_state = 42 -> fixes the random seed so the split is reproducible every run
# stratify=labels -> both splits will have the same sarcastic/non-sarcastic ratio,
#                    which matters if the classes are not perfectly balanced

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

print(f"Training examples : {len(X_train)}")
print(f"Test examples: {len(X_test)}")

## Tokenize

Transformers can't read raw strings. Tokenizer converts headlines into:
* input_ids - a list of integers, one per sub-work token (playing → "play"+"ing")
* attention_mask - 1 for real tokens, 0 for padding tokens.

GPUs process in batches, so padding is necessary. All sequences must be of the same legnth for tensors.

In [ ]:
# Load tokenizer that pairs with distilbert-base-uncased:
# distilbert - a compressed version of BERT, ~40% less parameters
# base - standard size (not the larger 'large' variant)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenize the entire training and test sets at once (batch tokenization is faster)
# * cut off headlines longer that max_length
# * pad shorter headlines with [PAD] tokens up to the longest
# * 128 tokens is generous for news headlines

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length = 128)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length = 128)

# Sanity check: decode the first training headline to confirm tokenizer round-trips
example_ids = train_encodings['input_ids'][0]
print("Token IDs for first training headline:")
print(example_ids)
print("\nDecoded back to text:")
print(tokenizer.decode(example_ids))


## Concept check
### What does the base model see?

According to the blog, the pre-training base model cannot distinguish emotionally and tonally different sentences yet. It was trained on general language, so its embeddings cluster all grammatically similar sentence close together.

We verify using cosine similarity on [CLS] token embeddings of two headlines: one sarcastic, one not.

* Cosine similarity = 1.0 means the vectors point in the same direction.
    The model thinks they are identical.
* 0.0 means perpendicular are unrelated. 
* Negative means opposite.

If the base model understood sarcasm, a sarcastic and a genuine headline should have low similarity:

In [ ]:
from transformers